
# Exp39 — Exact Pearson baseline sanity check on the frozen Exp33 test manifests

## Goal

Build the missing conventional retrieval baseline for the revised Table 1 under **exactly the same test pairs and memory protocol** used by the final PR comparison.

The paper-facing comparison becomes:

- **Pearson** — conventional centered-cosine / Pearson retrieval;
- **PRR-Stat (ablation)** — future-supervised predictive-relevance reranking within fixed Pearson Top-\(M\) support;
- **PRR (main)** — support expansion using Pearson Top-\(M\) \(\cup\) learned-representation Top-\(M\), followed by predictive-relevance reranking.

`PR-Hybrid` was a provisional internal name for what the current paper calls **PRR**. Likewise, `PR-Stat` maps to **PRR-Stat**, which is an ablation rather than the main method.

## Important

This notebook performs:

- **no training**,
- **no forecaster inference**,
- **no model loading**.

It only recomputes Pearson Top-10 retrieval on the frozen Exp33 test manifests and merges those values with the already validated Exp36R PR-Stat / PRR results.

The primary retrieval metric is `AnalogFutureMSE`.

**Aligned protocol note (final paper):** all six datasets use `memory_stride=8` for the long-horizon Pearson / PRR-Stat / full-PRR comparison. Stored outputs were cleared after the stride-alignment audit to avoid displaying stale stride-24 values.

In [ ]:

# Cell 1 — configuration

from pathlib import Path
import json, time, gc
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))

MANIFEST_ROOT = (
    ROOT / "results" / "exp33" / "final_eval_manifest"
)

EXP36R = ROOT / "results" / "exp36R"
PR_COMPARE_CSV = (
    EXP36R / "patchtst24_prr_stat_vs_prr.csv"
)

OUT = ROOT / "results" / "exp39_pearson_exact"
CACHE = OUT / "pearson_test_cache"

OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

DATASETS = (
    "ETTh1",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
)

HORIZONS = (96, 192, 336, 720)

CHANNELS = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

L = 96
TOP_K = 10
EPS = 1e-8

print("Manifest:", MANIFEST_ROOT)
print("PR table :", PR_COMPARE_CSV)
print("Output   :", OUT)

assert MANIFEST_ROOT.exists()
assert PR_COMPARE_CSV.exists()

print("[PASS] configuration")


In [ ]:

# Cell 2 — resolve the exact six raw datasets
#
# Same candidate paths used by Exp33 / Exp36R.

DATA_CANDIDATES = {
    "ETTh1": [
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETTh1.csv"),
    ],
    "Weather": [
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/dataset/weather.csv"),
    ],
    "Electricity": [
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity.csv"),
    ],
    "Traffic": [
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic.csv"),
    ],
    "Exchange": [
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate.csv"),
        Path("/data/dataset/exchange.csv"),
    ],
    "Solar": [
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/solar_AL.txt"),
    ],
}

dataset_paths = {}

for d, candidates in DATA_CANDIDATES.items():
    p = next((x for x in candidates if x.exists()), None)
    dataset_paths[d] = p
    print(f"{d:11s} -> {p}")

missing = [
    d for d,p in dataset_paths.items()
    if p is None
]

assert not missing, f"Missing raw datasets: {missing}"

print("[PASS] raw datasets 6/6")


In [ ]:

# Cell 3 — load raw data and reproduce Exp33/Exp36R boundaries + train-only normalization

def read_raw(dataset, path):
    if dataset == "Solar":
        try:
            df = pd.read_csv(path, header=None)
            if df.shape[1] == 1:
                df = pd.read_csv(
                    path,
                    header=None,
                    sep=r"\s+",
                )
        except Exception:
            df = pd.read_csv(
                path,
                header=None,
                sep=r"\s+",
            )
    else:
        df = pd.read_csv(path)
        if "date" in df.columns:
            df = df.drop(columns=["date"])

        # Keep OT last, matching the historical loader.
        if "OT" in df.columns:
            df = df[
                [x for x in df.columns if x != "OT"]
                + ["OT"]
            ]

    df = (
        df.apply(pd.to_numeric, errors="coerce")
        .dropna(axis=1, how="all")
        .replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )

    return df.to_numpy(np.float32)


def split_bounds(dataset, n):
    if dataset == "ETTh1":
        train_end = 12 * 30 * 24
        val_end = train_end + 4 * 30 * 24
    else:
        train_end = int(0.70 * n)
        val_end = n - int(0.20 * n)

    return int(train_end), int(val_end), int(n)


def memory_stride(dataset):
    # Final aligned long-horizon protocol: common stride=8 for all six datasets.
    return 8


DATA = {}

for dataset in DATASETS:
    raw = read_raw(
        dataset,
        dataset_paths[dataset],
    )

    assert raw.shape[1] == CHANNELS[dataset], (
        dataset,
        raw.shape,
    )

    tr, va, te = split_bounds(
        dataset,
        len(raw),
    )

    mu = raw[:tr].mean(
        axis=0
    ).astype(np.float32)

    sd = raw[:tr].std(
        axis=0,
        ddof=0,
    ).astype(np.float32)

    assert np.all(sd > 1e-6), dataset

    z = (
        (raw - mu[None,:])
        / sd[None,:]
    ).astype(np.float32)

    DATA[dataset] = {
        "z": z,
        "train_end": tr,
        "val_end": va,
        "test_end": te,
        "stride": memory_stride(dataset),
    }

    print(
        f"{dataset:11s}",
        raw.shape,
        "train=", tr,
        "val=", va,
        "stride=", memory_stride(dataset),
    )

print("[PASS] normalization / boundaries reproduced")


In [ ]:

# Cell 4 — exact Exp33 test-manifest loader and retrieval helpers

def load_test_manifest(dataset, horizon):
    p = (
        MANIFEST_ROOT
        / f"{dataset}_H{horizon}_test.npz"
    )

    assert p.exists(), p

    with np.load(p, allow_pickle=False) as z:
        a = np.asarray(
            z["anchor"],
            dtype=np.int64,
        )
        c = np.asarray(
            z["channel"],
            dtype=np.int64,
        )

    assert a.shape == c.shape
    assert len(a) > 0

    return a, c


def batch_pattern(x):
    """
    Centered cosine representation.
    For univariate windows this induces Pearson-correlation ranking.
    """
    x = np.asarray(
        x,
        dtype=np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    norm = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        norm > EPS,
        xc / np.maximum(norm, EPS),
        0.0,
    ).astype(np.float32)


def past_windows(z, channel, anchors):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    idx = (
        anchors[:,None]
        - L
        + np.arange(L)[None,:]
    )

    return z[
        idx,
        int(channel),
    ].astype(np.float32)


def future_residuals(z, channel, anchors, horizon):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    idx = (
        anchors[:,None]
        + np.arange(horizon)[None,:]
    )

    future = z[
        idx,
        int(channel),
    ].astype(np.float32)

    current = z[
        anchors - 1,
        int(channel),
    ].astype(np.float32)

    return (
        future
        - current[:,None]
    ).astype(np.float32)


# Preflight the 24 test manifests.
rows = []

for d in DATASETS:
    for h in HORIZONS:
        a,c = load_test_manifest(d,h)

        rows.append({
            "dataset": d,
            "horizon": h,
            "pairs": len(a),
            "channels": len(np.unique(c)),
            "anchor_min": int(a.min()),
            "anchor_max": int(a.max()),
        })

manifest_audit = pd.DataFrame(rows)

display(manifest_audit)

assert len(manifest_audit) == 24
assert (
    manifest_audit.channels
    == manifest_audit.dataset.map(CHANNELS)
).all()

print("[PASS] exact Exp33 test manifests 24/24")


In [ ]:

# Cell 5 — exact Pearson Top-10 AnalogFutureMSE
#
# Test memory contains train + validation.
# Candidate future must finish before val_end.
# Memory stride is exactly the historical protocol:
#   ETTh1 / Exchange: 8
#   others: 24
#
# This is the conventional Pattern/Pearson baseline.
#
# No training. No forecaster. No reranker.

def cache_path(dataset, horizon):
    return (
        CACHE
        / f"{dataset}_H{horizon}_pearson_test.npz"
    )


def valid_cached(dataset, horizon):
    p = cache_path(dataset, horizon)

    if not p.exists():
        return False

    try:
        ma, mc = load_test_manifest(
            dataset,
            horizon,
        )

        with np.load(p, allow_pickle=False) as z:
            a = np.asarray(z["anchor"], np.int64)
            c = np.asarray(z["channel"], np.int64)
            e = np.asarray(z["analog_mse"], np.float32)

        return (
            np.array_equal(a, ma)
            and np.array_equal(c, mc)
            and e.shape == ma.shape
            and np.isfinite(e).all()
        )

    except Exception:
        return False


def pearson_condition(dataset, horizon, query_chunk=128):
    if valid_cached(dataset, horizon):
        p = cache_path(dataset, horizon)

        with np.load(p, allow_pickle=False) as obj:
            e = np.asarray(
                obj["analog_mse"],
                dtype=np.float32,
            )

        return {
            "dataset": dataset,
            "horizon": int(horizon),
            "pearson_analog_future_mse":
                float(e.mean()),
            "pairs": int(len(e)),
            "cache": str(p),
            "reused": True,
        }

    data = DATA[dataset]
    z = data["z"]
    boundary = int(data["val_end"])
    stride = int(data["stride"])

    qa, qc = load_test_manifest(
        dataset,
        horizon,
    )

    pair_error = np.full(
        len(qa),
        np.nan,
        dtype=np.float32,
    )

    t0 = time.perf_counter()

    for ch in np.unique(qc):
        rows = np.flatnonzero(
            qc == ch
        )

        q_anchor = qa[rows]

        # Exact Exp36R test-memory support.
        mem_anchor = np.arange(
            L,
            boundary - horizon + 1,
            stride,
            dtype=np.int64,
        )

        assert len(mem_anchor) >= TOP_K

        # Leakage guard: all candidate futures end before/at boundary.
        assert (
            int(mem_anchor.max())
            + int(horizon)
            <= boundary
        )

        mem_past = past_windows(
            z,
            int(ch),
            mem_anchor,
        )

        mem_pattern = batch_pattern(
            mem_past
        )

        mem_future = future_residuals(
            z,
            int(ch),
            mem_anchor,
            horizon,
        )

        for left in range(
            0,
            len(rows),
            int(query_chunk),
        ):
            right = min(
                len(rows),
                left + int(query_chunk),
            )

            local_rows = rows[
                left:right
            ]

            q_a = qa[
                local_rows
            ]

            q_past = past_windows(
                z,
                int(ch),
                q_a,
            )

            q_pattern = batch_pattern(
                q_past
            )

            q_future = future_residuals(
                z,
                int(ch),
                q_a,
                horizon,
            )

            # Pearson ranking via centered cosine.
            score = (
                q_pattern
                @ mem_pattern.T
            )

            # Top-K highest Pearson correlations.
            top = np.argpartition(
                -score,
                TOP_K - 1,
                axis=1,
            )[:, :TOP_K]

            chosen_future = mem_future[
                top
            ]

            # AnalogFutureMSE:
            # mean candidate future error over Top-K and horizon.
            e = (
                (
                    chosen_future
                    - q_future[:,None,:]
                )
                ** 2
            ).mean(
                axis=(1,2)
            )

            pair_error[
                local_rows
            ] = e.astype(np.float32)

        print(
            f"  {dataset:11s} H={horizon:3d} "
            f"ch={int(ch):3d} "
            f"pairs={len(rows):4d} "
            f"memory={len(mem_anchor):5d}"
        )

    assert np.isfinite(pair_error).all()

    p = cache_path(
        dataset,
        horizon,
    )

    np.savez_compressed(
        p,
        anchor=qa.astype(np.int64),
        channel=qc.astype(np.int16),
        analog_mse=pair_error.astype(np.float32),
        memory_boundary=np.asarray(
            boundary,
            dtype=np.int64,
        ),
        memory_stride=np.asarray(
            stride,
            dtype=np.int64,
        ),
        top_k=np.asarray(
            TOP_K,
            dtype=np.int64,
        ),
        retrieval_name=np.asarray(
            "Pearson_centered_cosine",
        ),
    )

    sec = time.perf_counter() - t0

    return {
        "dataset": dataset,
        "horizon": int(horizon),
        "pearson_analog_future_mse":
            float(pair_error.mean()),
        "pairs": int(len(pair_error)),
        "cache": str(p),
        "reused": False,
        "seconds": float(sec),
    }


results = []

for dataset in DATASETS:
    for horizon in HORIZONS:
        print(
            "\n" + "="*90
        )
        print(
            "Pearson baseline:",
            dataset,
            "H=", horizon,
        )
        print(
            "="*90
        )

        row = pearson_condition(
            dataset,
            horizon,
        )

        results.append(row)

        print(
            f"Pearson AnalogFutureMSE = "
            f"{row['pearson_analog_future_mse']:.6f}"
        )

pearson24 = (
    pd.DataFrame(results)
    .sort_values(
        ["dataset","horizon"]
    )
    .reset_index(drop=True)
)

display(pearson24)

assert len(pearson24) == 24
assert pearson24.pearson_analog_future_mse.notna().all()

pearson24.to_csv(
    OUT / "pearson_24conditions.csv",
    index=False,
)

print("[PASS] Pearson baseline 24/24")


In [ ]:

# Cell 6 — merge with the already validated PR-Stat / PRR results

cmp = pd.read_csv(
    PR_COMPARE_CSV
)

cmp["horizon"] = (
    cmp["horizon"]
    .astype(int)
)

required = {
    "dataset",
    "horizon",
    "test_analog_future_mse",
    "learned_analog_future_mse",
}

missing = required - set(cmp.columns)

assert not missing, (
    f"Missing Exp36R columns: {sorted(missing)}"
)

table = pearson24[
    [
        "dataset",
        "horizon",
        "pearson_analog_future_mse",
    ]
].merge(
    cmp[
        [
            "dataset",
            "horizon",
            "test_analog_future_mse",
            "learned_analog_future_mse",
        ]
    ],
    on=["dataset","horizon"],
    how="inner",
)

assert len(table) == 24

table = table.rename(
    columns={
        "test_analog_future_mse":
            "pr_stat_analog_future_mse",
        "learned_analog_future_mse":
            "pr_hybrid_analog_future_mse",
    }
)

table[
    "pr_stat_gain_vs_pearson_pct"
] = (
    100.0
    * (
        table.pearson_analog_future_mse
        - table.pr_stat_analog_future_mse
    )
    / table.pearson_analog_future_mse
)

table[
    "pr_hybrid_gain_vs_pearson_pct"
] = (
    100.0
    * (
        table.pearson_analog_future_mse
        - table.pr_hybrid_analog_future_mse
    )
    / table.pearson_analog_future_mse
)

table[
    "pr_hybrid_gain_vs_pr_stat_pct"
] = (
    100.0
    * (
        table.pr_stat_analog_future_mse
        - table.pr_hybrid_analog_future_mse
    )
    / table.pr_stat_analog_future_mse
)

display(table)

table.to_csv(
    OUT / "table1_pearson_prstat_prhybrid.csv",
    index=False,
)

print("[PASS] exact 3-way Table-1 data created")


In [ ]:

# Cell 7 — scientific sanity summary

def wtl(gain, tol=1e-12):
    x = np.asarray(
        gain,
        dtype=float,
    )

    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
    }


comparisons = [
    (
        "PRR-Stat vs Pearson",
        table.pr_stat_gain_vs_pearson_pct,
    ),
    (
        "PRR vs Pearson",
        table.pr_hybrid_gain_vs_pearson_pct,
    ),
    (
        "PRR vs PRR-Stat",
        table.pr_hybrid_gain_vs_pr_stat_pct,
    ),
]

summary_rows = []

for name, gain in comparisons:
    c = wtl(gain)

    summary_rows.append({
        "comparison": name,
        "conditions": len(gain),
        **c,
        "mean_gain_pct":
            float(np.mean(gain)),
        "median_gain_pct":
            float(np.median(gain)),
    })

summary = pd.DataFrame(
    summary_rows
)

display(summary)

summary.to_csv(
    OUT / "table1_comparison_summary.csv",
    index=False,
)

print("\nDataset-level means")
dataset_summary = (
    table.groupby(
        "dataset",
        sort=False,
    )
    .agg(
        pearson_mse=(
            "pearson_analog_future_mse",
            "mean",
        ),
        pr_stat_mse=(
            "pr_stat_analog_future_mse",
            "mean",
        ),
        pr_hybrid_mse=(
            "pr_hybrid_analog_future_mse",
            "mean",
        ),
        pr_stat_gain_vs_pearson_pct=(
            "pr_stat_gain_vs_pearson_pct",
            "mean",
        ),
        pr_hybrid_gain_vs_pearson_pct=(
            "pr_hybrid_gain_vs_pearson_pct",
            "mean",
        ),
        pr_hybrid_gain_vs_pr_stat_pct=(
            "pr_hybrid_gain_vs_pr_stat_pct",
            "mean",
        ),
    )
    .reset_index()
)

display(dataset_summary)

dataset_summary.to_csv(
    OUT / "dataset_summary.csv",
    index=False,
)


In [ ]:

# Cell 8 — paper-ready compact table

paper = table.copy()

for col in [
    "pearson_analog_future_mse",
    "pr_stat_analog_future_mse",
    "pr_hybrid_analog_future_mse",
]:
    paper[col] = paper[col].map(
        lambda x: f"{x:.4f}"
    )

for col in [
    "pr_stat_gain_vs_pearson_pct",
    "pr_hybrid_gain_vs_pearson_pct",
    "pr_hybrid_gain_vs_pr_stat_pct",
]:
    paper[col] = paper[col].map(
        lambda x: f"{x:+.2f}%"
    )

paper = paper.rename(
    columns={
        "dataset": "Dataset",
        "horizon": "H",
        "pearson_analog_future_mse":
            "Pearson",
        "pr_stat_analog_future_mse":
            "PRR-Stat (ablation)",
        "pr_hybrid_analog_future_mse":
            "PRR (main)",
        "pr_stat_gain_vs_pearson_pct":
            "PRR-Stat vs Pearson",
        "pr_hybrid_gain_vs_pearson_pct":
            "PRR vs Pearson",
        "pr_hybrid_gain_vs_pr_stat_pct":
            "PRR vs PRR-Stat",
    }
)

display(paper)

paper.to_csv(
    OUT / "paper_table1_compact.csv",
    index=False,
)

print("[PASS] paper-ready table saved")


In [ ]:

# Cell 9 — final paper-facing readout

stat_gain = (
    table.pr_stat_gain_vs_pearson_pct
)

hybrid_gain = (
    table.pr_hybrid_gain_vs_pearson_pct
)

hybrid_stat_gain = (
    table.pr_hybrid_gain_vs_pr_stat_pct
)

final = {
    "experiment":
        "Exp39 Pearson exact sanity",

    "protocol":
        "Exp33 exact test manifests",

    "conditions": 24,

    "pearson_definition":
        "centered cosine = Pearson ranking",

    "training_performed": False,
    "forecaster_inference_performed": False,

    "pr_stat_vs_pearson":
        {
            **wtl(stat_gain),
            "mean_gain_pct":
                float(stat_gain.mean()),
            "median_gain_pct":
                float(stat_gain.median()),
        },

    "pr_hybrid_vs_pearson":
        {
            **wtl(hybrid_gain),
            "mean_gain_pct":
                float(hybrid_gain.mean()),
            "median_gain_pct":
                float(hybrid_gain.median()),
        },

    "pr_hybrid_vs_pr_stat":
        {
            **wtl(hybrid_stat_gain),
            "mean_gain_pct":
                float(hybrid_stat_gain.mean()),
            "median_gain_pct":
                float(hybrid_stat_gain.median()),
        },
}

(
    OUT / "exp39_summary.json"
).write_text(
    json.dumps(
        final,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        final,
        indent=2,
    )
)

print("\n" + "="*100)
print("EXP39 COMPLETE")
print("="*100)
print(
    "Use Table 1 as: "
    "Pearson | PRR-Stat (ablation) | PRR (main)"
)
print(
    "No training or forecaster inference was performed."
)
print("Saved under:", OUT)
